# Credit Risk Classification

A leakage-resistant comparison of classifiers for an imbalanced credit-risk dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier

RANDOM_STATE = 20
DATA_PATH = Path("../data/credit_risk_dataset.csv")
dataset = pd.read_csv(DATA_PATH)
dataset.head()


## Data quality and exploratory analysis

In [ ]:
print("Shape:", dataset.shape)
display(dataset.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("Duplicate rows:", dataset.duplicated().sum())

numeric = dataset.select_dtypes(include="number")
plt.figure(figsize=(11, 8))
sns.heatmap(numeric.corr(), cmap="coolwarm", center=0)
plt.title("Numeric Feature Correlations")
plt.tight_layout()
plt.show()

sns.countplot(data=dataset, x="loan_status")
plt.title("Target Class Distribution")
plt.show()


## Preprocessing and split

Duplicates and implausible ages are removed. The split happens before any fitted preprocessing or resampling. Missing-value imputation, one-hot encoding, scaling, and SMOTE are fitted on training data only through each model pipeline.

In [ ]:
dataset = dataset.drop_duplicates().copy()
dataset = dataset.loc[dataset["person_age"].between(18, 100)]

X = dataset.drop(columns="loan_status")
y = dataset["loan_status"].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=1 / 3, random_state=RANDOM_STATE, stratify=y_temp
)

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = SklearnPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = SklearnPipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])


## Model comparison

Validation ROC AUC is computed from predicted probabilities, not hard class labels. Recall and F1 are reported because missed defaults can be more important than overall accuracy.

In [ ]:
estimators = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE),
}

results = {}
for name, estimator in estimators.items():
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", estimator),
    ])
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_val)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    results[name] = {
        "model": model,
        "precision": precision_score(y_val, prediction, zero_division=0),
        "recall": recall_score(y_val, prediction, zero_division=0),
        "f1": f1_score(y_val, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y_val, probability),
    }

validation_results = pd.DataFrame({
    name: {key: value for key, value in result.items() if key != "model"}
    for name, result in results.items()
}).T.sort_values("roc_auc", ascending=False)
display(validation_results)


## Final holdout evaluation

The test set is used once, after model selection on validation ROC AUC. The decision threshold remains 0.5; production use would require cost-sensitive threshold selection, calibration, fairness review, and temporal validation.

In [ ]:
best_model_name = validation_results.index[0]
best_model = results[best_model_name]["model"]
test_probability = best_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

print("Selected model:", best_model_name)
print("Test ROC AUC:", round(roc_auc_score(y_test, test_probability), 3))
print(classification_report(y_test, test_prediction, digits=3, zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test, test_prediction),
                     index=["Actual 0", "Actual 1"],
                     columns=["Predicted 0", "Predicted 1"]))

plt.figure(figsize=(8, 6))
for name, result in results.items():
    probability = result["model"].predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, probability)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_val, probability):.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()
